In [1]:
from pathlib import Path 

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import osmnx as ox


In [2]:
from adjustText import adjust_text

In [3]:
import ipywidgets as widgets

In [4]:
print(Path.cwd())

d:\Wu\2026\Project Portfolio\002 Project\grocery_stores_access\notebooks


In [5]:
PROCESSED_DIR = Path("../data/processed")

In [6]:
county_boundary = gpd.read_file(PROCESSED_DIR / "county_boundary.gpkg")
stores = gpd.read_file(PROCESSED_DIR / "grocery_store_nodes.gpkg")
G = ox.load_graphml(PROCESSED_DIR / "kootenai_drive.graphml")

In [7]:
nodes, edges = ox.graph_to_gdfs(G)
roads = edges

In [8]:
cities = gpd.read_file(PROCESSED_DIR / "kootenai_cities.gpkg")

In [9]:
all_service_areas = gpd.read_file(
    PROCESSED_DIR / "grocery_service_areas.gpkg"
)

In [10]:
all_service_areas = all_service_areas.rename(columns={"name": "store"})

In [11]:
all_service_areas.head()

,store,city,minutes,geometry
0,Grocery Outlet,Coeur d'Alene,10,"POLYGON ((503268.061 5281532.66, 503190.023 52..."
1,Grocery Outlet,Coeur d'Alene,20,"POLYGON ((498134.235 5278426.094, 498119.326 5..."
2,Grocery Outlet,Coeur d'Alene,30,"POLYGON ((496235.593 5253261.756, 496216.463 5..."
3,Little Town Market,Athol,10,"POLYGON ((512057.612 5309792.183, 512055.879 5..."
4,Little Town Market,Athol,20,"POLYGON ((506809.835 5291444.637, 506738.984 5..."


In [12]:
all_service_areas.shape

(90, 4)

In [13]:
all_service_areas["minutes"].value_counts()

minutes
10    30
20    30
30    30
Name: count, dtype: int64

In [14]:
all_service_areas = all_service_areas.to_crs(roads.crs)
cities = cities.to_crs(roads.crs)
stores = stores.to_crs(roads.crs)
county_boundary = county_boundary.to_crs(roads.crs)

In [15]:
print(roads.crs)
print(all_service_areas.crs)
print(cities.crs)
print(stores.crs)
print(county_boundary.crs)

epsg:4326
epsg:4326
epsg:4326
EPSG:4326
epsg:4326


In [16]:
service10 = all_service_areas[
    all_service_areas["minutes"] == 10
]

In [17]:
service20 = all_service_areas[
    all_service_areas["minutes"] == 20
]

In [18]:
service30 = all_service_areas[
    all_service_areas["minutes"] == 30
]

## Filter Map Display
### One filter based on drive time: 10min, 20min, and 30min
### The other filter based on store name

In [19]:
all_service_areas.columns

Index(['store', 'city', 'minutes', 'geometry'], dtype='str')

In [20]:
# Create Dropdown options
stores_list = (
    ["All Stores"]
    + sorted(all_service_areas["store"].unique())
)

minutes_list = [
    "All",
    10,
    20,
    30
]

In [21]:
store_selector = widgets.SelectMultiple(
    options=sorted(all_service_areas["store"].unique()),
    value=(),     # nothing selected initially
    description="Stores:"
)

In [22]:
minute_dropdown = widgets.Dropdown(
    options=minutes_list,
    value="All",
    description="Minutes:"
)

In [23]:
display(
    store_selector,
    minute_dropdown
)

SelectMultiple(description='Stores:', options=('Albertsons', 'C & C Grocery', 'Flour Mill Natural Foods', 'Fre…

Dropdown(description='Minutes:', options=('All', 10, 20, 30), value='All')

In [24]:
stores = stores.rename(columns={"name": "store"})

In [25]:
def draw_map(selected_minutes, selected_stores):  
    filtered = all_service_areas.copy()
    if len(selected_stores) > 0:
        filtered = filtered[
            filtered["store"].isin(selected_stores)
        ]

    if selected_minutes != "All":
        filtered = filtered[
            filtered["minutes"] == selected_minutes
        ]
    filtered_stores = stores.copy()
    if len(selected_stores) > 0:
        filtered_stores = filtered_stores[
            filtered_stores["store"].isin(selected_stores)
        ]

    fig, ax = plt.subplots(figsize=(10, 8))

    county_boundary.plot(ax=ax, color="lightgray", linewidth=1, zorder=1)
    roads.plot(ax=ax, color="darkgray", linewidth=0.5, zorder=2)

    filtered.plot(ax=ax, alpha=0.4, zorder=3)
    
    filtered_stores.plot(
        ax=ax,
        marker="*",
        color="red",
        markersize=20,
        zorder=4
    )

    texts = []
    for _, row in cities.iterrows():
        label_point = row.geometry.representative_point()

        texts.append(
            ax.text(
                label_point.x,
                label_point.y,
                row["city"],
                fontsize=9
            )
        )
    adjust_text(texts)

    ax.set_title(f"Service Areas {selected_stores}-{selected_minutes}min")
    plt.show()

In [26]:
ui = widgets.interactive(
    draw_map,
    selected_minutes=minute_dropdown,
    selected_stores=store_selector
)

display(ui)

interactive(children=(Dropdown(description='Minutes:', options=('All', 10, 20, 30), value='All'), SelectMultip…